# Exercise 4: decision trees and random forests with a bag of k-mers

Return to the artificial TFBS dataset from Exercise 1.
Compare a **decision tree** and a **random forest with 100 trees** using **k = 1 through 7**.
Use one k-mer length at a time: there are **no mixed k-mers**.

Keep the same **70% training / 20% validation / 10% test split** as Exercise 2.
Using the same CSV and seed gives the same sequences in each split.
Fit vocabularies and models on training sequences only.
Choose the best k **separately for each model** using validation ROC-AUC.
Evaluate those two selected models on the test set only at the end.

Walk through both models at **k = 4** before the comparison loop.
The class labels remain 1 = planted motif, 0 = no planted motif.

In Colab, upload the Exercise 1 CSV using the Files sidebar and set `ifile` below.
No other project files are needed. If libraries are missing, install them with
`%pip install numpy pandas scikit-learn matplotlib` in a separate cell.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, log_loss,
    RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay,
)

ifile = "exercise_1_output/dataset.csv"
seed = 42
threshold = 0.5
k_values = range(1, 8)


## 1. Load the Exercise 1 dataset

The CSV has two columns: `sequence` and `label`. Label 1 means a planted motif;
label 0 means no planted motif. In Colab, an uploaded file named `dataset.csv`
can be read by changing `ifile` to `"dataset.csv"` above.

In [ ]:
dataset = pd.read_csv(ifile)

assert dataset["sequence"].str.fullmatch("[ACGT]{100}").all()
assert dataset["label"].isin([0, 1]).all()
assert dataset["label"].nunique() == 2
assert dataset["label"].value_counts().nunique() == 1

print("Sequences:", len(dataset))
print(dataset["label"].value_counts())
dataset.head()


## 2. Randomly select test and validation sequences for each label

Each label has 10,000 sequences. For each label:

1. Randomly select **1,000 sequences (10%) for test**.
2. Remove them, then randomly select **2,000 sequences (20% of the original 10,000) for validation**.
3. Keep the remaining **7,000 sequences (70%) for training**.

`groupby("label").sample(...)` selects the requested number from each label.
`drop(...)` removes selected rows, so the splits cannot overlap.
We calculate both counts from the original class size, not from what remains.
This dataset has equal numbers of positives and negatives.

Use this **same split for every k**. Fit the vocabulary and model on training sequences only.


In [ ]:
sequences_per_label = len(dataset) // 2
n_test = int(0.10 * sequences_per_label)
n_validation = int(0.20 * sequences_per_label)

test_data = dataset.groupby("label").sample(n=n_test, random_state=seed)
remaining_data = dataset.drop(test_data.index)
validation_data = remaining_data.groupby("label").sample(n=n_validation, random_state=seed)
train_data = remaining_data.drop(validation_data.index)

y_train = train_data["label"]
y_validation = validation_data["label"]

print("Training:", len(train_data))
print("Validation:", len(validation_data))
print("Test (reserved):", len(test_data))


## 3. Moving window -> bag of k-mers

A k-mer is a DNA substring of length k. Move a window one base at a time,
including overlapping windows. For `ACACAC` and k = 2, the windows are
`AC`, `CA`, `AC`, `CA`, `AC`. The bag contains counts: AC = 3, CA = 2.

We keep **counts**, but discard where each k-mer occurred and the order of the windows.
For a 100-nt sequence, there are 100 - k + 1 windows.

`CountVectorizer(analyzer="char", ngram_range=(k, k))` performs this moving-window counting.
It stores a sparse matrix: rows are sequences, columns are training-observed k-mers,
and entries are counts. An unseen k-mer is ignored when transforming validation or test sequences.
We do not merge reverse complements or use TF-IDF.

In [ ]:
example = "ACACAC"
example_k = 2
windows = []

for start in range(len(example) - example_k + 1):
    windows.append(example[start:start + example_k])

print("Moving windows:", windows)
pd.Series(windows).value_counts()


## 4. Walk through both models: k = 4

Change `example_k` below if you want to demonstrate a different k.
Count the k-mers once and give the **same count matrix to both models**.

- A **decision tree** asks a series of questions about k-mer counts.
- A **random forest** averages the positive-class probabilities of 100 trees.
  Each tree uses a bootstrap sample of training sequences, and considers a random subset of features at each split.

Keep the default Gini split criterion. Do not tune tree depth or other model settings here;
k is the only hyperparameter we compare. The trees have no explicit maximum depth.
Unlike logistic regression, these models do not fit coefficients by minimizing binary cross-entropy.
Log loss is still useful as an evaluation metric: confident wrong predictions receive a large penalty.
A fully grown single tree can produce probabilities of 0 or 1, so its log loss can be large.

The cell below fits both models, prints the same eight validation metrics as Exercise 2,
and shows ROC curves, precision-recall curves, and confusion matrices.
It does not use the test set. The next loop repeats this workflow for k = 1 through 7.


In [ ]:
example_k = 4

example_vectorizer = CountVectorizer(
    analyzer="char", ngram_range=(example_k, example_k), lowercase=False, dtype=np.int32,
)
x_example_train = example_vectorizer.fit_transform(train_data["sequence"])
x_example_validation = example_vectorizer.transform(validation_data["sequence"])
print("Training count matrix:", x_example_train.shape)
print("Validation count matrix:", x_example_validation.shape)

example_models = {
    "Decision tree": DecisionTreeClassifier(random_state=seed),
    "Random forest": RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1),
}
example_probabilities = {}
rows = []
fig, axes = plt.subplots(2, 3, figsize=(17, 10))

for row, (name, model) in enumerate(example_models.items()):
    model.fit(x_example_train, y_train)
    probabilities = model.predict_proba(x_example_validation)[:, 1]
    example_probabilities[name] = probabilities
    predictions = (probabilities >= threshold).astype(int)
    rows.append({
        "model": name,
        "k": example_k,
        "features": x_example_train.shape[1],
        "accuracy": accuracy_score(y_validation, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_validation, predictions),
        "precision": precision_score(y_validation, predictions, zero_division=0),
        "recall": recall_score(y_validation, predictions, zero_division=0),
        "f1": f1_score(y_validation, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_validation, probabilities),
        "average_precision": average_precision_score(y_validation, probabilities),
        "log_loss": log_loss(y_validation, probabilities),
    })

    RocCurveDisplay.from_predictions(y_validation, probabilities, ax=axes[row, 0], name=name)
    axes[row, 0].plot([0, 1], [0, 1], "k--")
    PrecisionRecallDisplay.from_predictions(y_validation, probabilities, ax=axes[row, 1], name=name)
    axes[row, 1].axhline(y_validation.mean(), color="black", linestyle="--")
    ConfusionMatrixDisplay.from_predictions(
        y_validation, predictions, labels=[0, 1], display_labels=["Negative", "Positive"],
        ax=axes[row, 2], colorbar=False, cmap="Blues",
    )
    axes[row, 0].set_title(f"{name}, k={example_k}: validation ROC")
    axes[row, 1].set_title(f"{name}, k={example_k}: validation precision-recall")
    axes[row, 2].set_title(f"{name}, k={example_k}: validation confusion matrix")

example_results = pd.DataFrame(rows).set_index(["model", "k"])
print("Validation metrics:")
print(example_results.round(4).to_string())
plt.tight_layout()
plt.show()


### Visualize the decision tree at k=4

Show only the first three levels of the fitted decision tree so the plot stays readable.
This limits the drawing, not the trained model. The `(...)` boxes hide deeper branches.

Each split asks whether a k-mer count is below a threshold (left: yes; right: no).
For example, `ACGT <= 0.5` separates sequences without ACGT from sequences containing it.
`samples` is the number of training sequences at a node; `value` gives the negative and positive counts.
The color indicates the predicted class.


In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(20, 9))
plot_tree(
    example_models["Decision tree"],
    max_depth=2,
    feature_names=example_vectorizer.get_feature_names_out(),
    class_names=["Negative", "Positive"],
    filled=True,
    impurity=False,
    fontsize=10,
)
plt.title("Decision tree at k=4: first three levels")
plt.tight_layout()
plt.show()


## 5. Compare both models for k = 1 through 7

For each k, fit one vocabulary on training sequences and transform validation sequences.
Both models receive the same sparse count matrices. Do not convert them to dense arrays.
Larger k can produce many more features and make fitting slower.
The forest uses 100 trees and `n_jobs=-1` to use the available CPU cores.

Keep the model settings fixed. Store validation probabilities for each model and k.
Keep the training-fitted model and vocabulary with the highest **validation ROC-AUC**
for each model separately. If scores tie exactly, keep the smaller k.
Choose this rule before looking at the test results.


In [ ]:
model_names = ["Decision tree", "Random forest"]
validation_probabilities = {"Decision tree": {}, "Random forest": {}}
feature_counts = {}
best_validation_auc = {"Decision tree": -1, "Random forest": -1}
best_k = {}
best_models = {}
best_vectorizers = {}

for k in k_values:
    vectorizer = CountVectorizer(
        analyzer="char", ngram_range=(k, k), lowercase=False, dtype=np.int32,
    )
    x_train = vectorizer.fit_transform(train_data["sequence"])
    x_validation = vectorizer.transform(validation_data["sequence"])
    feature_counts[k] = x_train.shape[1]

    models = {
        "Decision tree": DecisionTreeClassifier(random_state=seed),
        "Random forest": RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1),
    }
    for name, model in models.items():
        model.fit(x_train, y_train)
        probabilities = model.predict_proba(x_validation)[:, 1]
        validation_probabilities[name][k] = probabilities
        validation_auc = roc_auc_score(y_validation, probabilities)

        if validation_auc > best_validation_auc[name]:
            best_validation_auc[name] = validation_auc
            best_k[name] = k
            best_models[name] = model
            best_vectorizers[name] = vectorizer

        print(f"{name} | k={k:2d} | features={feature_counts[k]:7,d} | validation ROC-AUC={validation_auc:.4f}")


## 6. Compare validation metrics for every k

- **Accuracy:** fraction of correct class predictions.
- **Balanced accuracy:** mean recall across the two classes.
- **Precision:** among predicted positives, the fraction that are positive.
- **Recall:** among actual positives, the fraction predicted positive.
- **F1:** harmonic mean of precision and recall.
- **ROC-AUC:** ranking quality across thresholds; higher is better, and 0.5 is chance-level ranking.
- **Average precision (AP):** a summary of the precision-recall curve; higher is better. This is not trapezoidal PR-AUC.
- **Log loss:** error in the predicted probabilities; lower is better.

Accuracy, balanced accuracy, precision, recall, F1, and confusion matrices use the fixed
threshold of 0.5. ROC-AUC and AP use probabilities across thresholds, not the class calls.
If a model makes no positive calls, report precision as 0 (`zero_division=0`).

In [ ]:
rows = []

for name in model_names:
    for k in k_values:
        probabilities = validation_probabilities[name][k]
        predictions = (probabilities >= threshold).astype(int)
        rows.append({
            "model": name,
            "k": k,
            "features": feature_counts[k],
            "accuracy": accuracy_score(y_validation, predictions),
            "balanced_accuracy": balanced_accuracy_score(y_validation, predictions),
            "precision": precision_score(y_validation, predictions, zero_division=0),
            "recall": recall_score(y_validation, predictions, zero_division=0),
            "f1": f1_score(y_validation, predictions, zero_division=0),
            "roc_auc": roc_auc_score(y_validation, probabilities),
            "average_precision": average_precision_score(y_validation, probabilities),
            "log_loss": log_loss(y_validation, probabilities),
        })

validation_results = pd.DataFrame(rows).set_index(["model", "k"])
validation_results.round(4)


## 7. Compare validation ROC and precision-recall curves

Each row is one model; each curve is one k. All curves use the same validation sequences.
The ROC diagonal is chance-level ranking. The precision-recall reference line is the positive-class fraction.
The legends show ROC-AUC and average precision.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for row, name in enumerate(model_names):
    for k in k_values:
        RocCurveDisplay.from_predictions(
            y_validation, validation_probabilities[name][k], name=f"k={k}", ax=axes[row, 0],
        )
        PrecisionRecallDisplay.from_predictions(
            y_validation, validation_probabilities[name][k], name=f"k={k}", ax=axes[row, 1],
        )
    axes[row, 0].plot([0, 1], [0, 1], "k--", label="Chance")
    axes[row, 1].axhline(y_validation.mean(), color="black", linestyle="--", label="Class prevalence")
    axes[row, 0].set_title(f"{name}: validation ROC curves")
    axes[row, 1].set_title(f"{name}: validation precision-recall curves")
    axes[row, 0].legend(fontsize=8)
    axes[row, 1].legend(fontsize=8)

plt.tight_layout()
plt.show()


## 8. Compare validation confusion matrices

Show one grid for each model, with one matrix per k at the fixed 0.5 threshold.
Rows are actual labels and columns are predicted labels.
The shared color scale makes the matrices comparable.


In [ ]:
for name in model_names:
    fig, axes = plt.subplots(2, 4, figsize=(16, 7))
    for k, ax in zip(k_values, axes.flat):
        predictions = (validation_probabilities[name][k] >= threshold).astype(int)
        ConfusionMatrixDisplay.from_predictions(
            y_validation, predictions, labels=[0, 1], display_labels=["Negative", "Positive"],
            ax=ax, colorbar=False, cmap="Blues",
            im_kw={"vmin": 0, "vmax": y_validation.value_counts().max()},
        )
        ax.set_title(f"Validation, k={k}")
    axes[1, 3].set_visible(False)
    fig.suptitle(name)
    plt.tight_layout()
    plt.show()


## 9. Select k separately for each model

Use the largest validation ROC-AUC for each model, not whichever metric looks best afterwards.
Discuss the feature counts, scores, and curves before continuing.
Both selected models remain fitted on the 70% training split only; do not refit on validation data.


In [ ]:
for name in model_names:
    print(name, "selected k:", best_k[name])
    print("Validation ROC-AUC:", round(best_validation_auc[name], 4))
    print(validation_results.loc[[(name, best_k[name])]].round(4).to_string())
    print()


## 10. STOP before the test evaluation

Finish choosing one k per model and discussing validation results **before running the cell below**.
**Run all also runs this final test evaluation.** Run cells individually when teaching the split boundaries.

Evaluate the selected decision tree and selected random forest on the same reserved test sequences.
Each uses its own training-fitted vocabulary. Do not fit, select, or change anything using test results.


In [ ]:
y_test = test_data["label"]
test_probabilities = {}
rows = []
fig, axes = plt.subplots(2, 3, figsize=(17, 10))

for row, name in enumerate(model_names):
    x_test = best_vectorizers[name].transform(test_data["sequence"])
    probabilities = best_models[name].predict_proba(x_test)[:, 1]
    test_probabilities[name] = probabilities
    predictions = (probabilities >= threshold).astype(int)
    rows.append({
        "model": name,
        "k": best_k[name],
        "accuracy": accuracy_score(y_test, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions, zero_division=0),
        "recall": recall_score(y_test, predictions, zero_division=0),
        "f1": f1_score(y_test, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_test, probabilities),
        "average_precision": average_precision_score(y_test, probabilities),
        "log_loss": log_loss(y_test, probabilities),
    })
    RocCurveDisplay.from_predictions(y_test, probabilities, ax=axes[row, 0], name=name)
    axes[row, 0].plot([0, 1], [0, 1], "k--")
    PrecisionRecallDisplay.from_predictions(y_test, probabilities, ax=axes[row, 1], name=name)
    axes[row, 1].axhline(y_test.mean(), color="black", linestyle="--")
    ConfusionMatrixDisplay.from_predictions(
        y_test, predictions, labels=[0, 1], display_labels=["Negative", "Positive"],
        ax=axes[row, 2], colorbar=False, cmap="Blues",
    )
    axes[row, 0].set_title(f"{name}, k={best_k[name]}: test ROC")
    axes[row, 1].set_title(f"{name}, k={best_k[name]}: test precision-recall")
    axes[row, 2].set_title(f"{name}, k={best_k[name]}: test confusion matrix")

test_results = pd.DataFrame(rows).set_index(["model", "k"])
print("Final test metrics:")
print(test_results.round(4).to_string())
plt.tight_layout()
plt.show()


## Discussion

How did a single tree compare with averaging 100 trees? Did the preferred k change?
Compare ROC-AUC with log loss: a model can rank sequences usefully while giving overly confident probabilities.
These results concern the simulated dataset, not real promoter prediction.
